In [9]:
# Enables IPython autoreload (two magic commands, the second takes a numeric argument)
%load_ext autoreload
%autoreload 2

import logging
import os
import time

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s  - %(name)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.6f}".format)


logger.info("Notebook initialized")

2026-05-07 13:23:43,974  - __main__ - INFO - Notebook initialized


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

## Final pipeline using `src/data.py`

The above cells walked through the exploration step-by-step. The same pipeline is now packaged into two functions in `src/data.py`. The cells below verify that the imported functions produce the same output as the inline exploration.

In [10]:
from src.data import compute_returns, download_prices

prices = download_prices()
returns = compute_returns(prices)

print("Daily prices shape:", prices.shape)
print("Monthly returns shape:", returns.shape)
print("\nFirst 3 returns:")
print(returns.head(3))
print("\nLast 3 returns:")
print(returns.tail(3))

2026-05-07 13:23:43,994  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-05-07 13:23:44,004  - src.data - INFO - Computed monthly returns: (120, 8)


Daily prices shape: (2538, 8)
Monthly returns shape: (120, 8)

First 3 returns:
                 SPY      GOVT     EEMV       CME       BR      CBOE  \
date                                                                   
2015-01-31 -0.029629  0.029423 0.011831 -0.037789 0.039194  0.016556   
2015-02-28  0.056205 -0.017439 0.025305  0.124619 0.109189 -0.065708   
2015-03-31 -0.015745  0.006021 0.004426 -0.007544 0.038856 -0.043728   

                 ICE       ACN  
date                            
2015-01-31 -0.061836 -0.059120  
2015-02-28  0.144024  0.071403  
2015-03-31 -0.006068  0.040653  

Last 3 returns:
                 SPY      GOVT      EEMV      CME        BR      CBOE  \
date                                                                    
2024-10-31 -0.008924 -0.024286 -0.037161 0.021346 -0.019393  0.042466   
2024-11-30  0.059634  0.008489 -0.008945 0.056088  0.119321  0.013626   
2024-12-31 -0.024100  0.006945 -0.007585 0.004852 -0.038463 -0.094742   

           

In [11]:
from src.data import compute_returns, download_prices
from src.stats import arithmetic_mean, geometric_mean

prices = download_prices()
returns = compute_returns(prices)

print("Arithmetic mean (monthly):")
print(arithmetic_mean(returns))
print("\nGeometric mean (monthly):")
print(geometric_mean(returns))

2026-05-07 13:23:44,025  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-05-07 13:23:44,030  - src.data - INFO - Computed monthly returns: (120, 8)


Arithmetic mean (monthly):
SPY    0.011216
GOVT   0.000917
EEMV   0.002965
CME    0.012852
BR     0.016831
CBOE   0.012521
ICE    0.013060
ACN    0.015002
dtype: float64

Geometric mean (monthly):
SPY    0.010243
GOVT   0.000815
EEMV   0.002326
CME    0.011446
BR     0.014829
CBOE   0.010574
ICE    0.011327
ACN    0.012883
dtype: float64


In [13]:
from src.backtest import run_backtest
from src.data import compute_returns, download_prices

prices = download_prices()
returns = compute_returns(prices)

results = run_backtest(returns, train_fraction=0.6)
results.round(4)

2026-05-07 13:24:22,410  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-05-07 13:24:22,416  - src.data - INFO - Computed monthly returns: (120, 8)
2026-05-07 13:24:22,416  - src.backtest - INFO - Split 120 observations: train=72 (2015-01-31 to 2020-12-31), test=48 (2021-01-31 to 2024-12-31)
2026-05-07 13:24:22,421  - src.sharpe - INFO - Tangency (QP): Sharpe=0.5101, return=0.0063, volatility=0.0123
2026-05-07 13:24:22,426  - src.optimizer - INFO - cvxpy MVO converged: variance=0.000088, return=0.003976, status=optimal
2026-05-07 13:24:22,429  - src.backtest - INFO - Backtest complete; results:
              cumulative_return  annualized_return  annualized_volatility  \
tangency               0.059100           0.017800               0.083600   
min_variance           0.003500           0.003100               0.067800   
equal_weight           0.468400           0.105300               0.134400   

    

,cumulative_return,annualized_return,annualized_volatility,annualized_sharpe,max_drawdown,final_value
tangency,0.059100,0.017800,0.083600,0.212900,-0.177800,1.059100
min_variance,0.003500,0.003100,0.067800,0.046200,-0.155400,1.003500
equal_weight,0.468400,0.105300,0.134400,0.783200,-0.220100,1.468400
